#### **What is a Model in dbt?**

> A **dbt model** is **just a SQL SELECT query saved in a** <mark>.sql</mark> **file** that dbt runs and **turns into a table or view** in your data warehouse.

That’s it.

No magic SQL.

No stored procedures.

No complex configs at first.


📌 Think like this:

- You write **SELECT SQL**

- dbt runs it

- dbt **creates or updates** a table/view in Snowflake/BigQuery/Redshift/etc.


---------


Each model:

- Has **one responsibility**

- Is **reusable**

- Is **version-controlled**

This is **software engineering applied to SQL.**

-----------

#### **What happens when you run a model?**

When you run:

In [ ]:
dbt run

dbt does this:

- Reads your `.sql` file

- Compiles it (resolves ref(), configs, etc.)

- Runs the SQL in your warehouse

- Creates a **table or view**

-------

#### **First Simple Example (No dbt magic yet)**

File: `models/listings.sql`


In [ ]:
SELECT
    id,
    name,
    price
FROM raw.raw_listings

**What dbt creates in Snowflake**

`DEV.LISTINGS`

👉 This is now a dbt model

--------

#### **Materializations (What dbt creates)**

Every model must be **materialized** as something:


| Materialization  | What it becomes      |
| ---------------- | -------------------- |
| `view` (default) | View                 |
| `table`          | Table                |
| `incremental`    | Append/update table  |
| `ephemeral`      | Temporary (CTE only) |



> **Default behavior** : If you write **nothing**, dbt creates a **VIEW**.

---------

#### **Model with Explicit Materialization**

In [ ]:
{{ config(materialized='table') }}

SELECT
    id,
    name,
    price
FROM raw.raw_listings

Now dbt creates:

`DEV.LISTINGS   (TABLE)`

----------


**The MOST IMPORTANT RULE:** `ref()`

**❌ Bad (hard-coded)**

In [ ]:
SELECT * FROM dev.src_listings

**✅ Correct (dbt way)**

In [ ]:
SELECT * FROM {{ ref('src_listings') }}

#### **Why ref() is critical**

| Benefit             | Why it matters               |
| ------------------- | ---------------------------- |
| Dependency tracking | dbt knows order              |
| Environment safety  | Dev/Prod works automatically |
| Lineage             | dbt draws DAG                |
| Refactoring         | Rename safely                |


--------

#### **Incremental Model (Production-Level)**

**Example: `fct_reviews.sql`**

In [ ]:
{{ config(
    materialized='incremental',
    unique_key='review_id'
) }}

SELECT
    review_id,
    listing_id,
    review_date,
    review_text
FROM {{ ref('src_reviews') }}

{% if is_incremental() %}
WHERE review_date > (SELECT max(review_date) FROM {{ this }})
{% endif %}


**What this does**

- First run → full load

- Next runs → only new data

- Saves **time + cost**

-------------

#### **`this` keyword (Important)**

In [ ]:
{{ this }}

Means: `DEV.FCT_REVIEWS`

Used only inside incremental models.

---

**How dbt models fit into analytics layers**

| Layer     | Purpose        | Example        |
| --------- | -------------- | -------------- |
| Source    | Raw data       | `raw_listings` |
| Staging   | Clean & rename | `stg_listings` |
| Dimension | Who / What     | `dim_hosts`    |
| Fact      | Events         | `fct_reviews`  |

----------

**What makes dbt models powerful (Interview-level clarity)**

✔ Version controlled

✔ Modular

✔ Testable

✔ Documented

✔ Lineage-aware

✔ Environment-safe

This is why **dbt is loved in data engineering.**

-------------

> **A dbt model = one clean, reusable SELECT statement that represents a business concept**